In [ ]:
# eff map
import pandas as pd
import numpy as np
import math
from scipy.interpolate import RegularGridInterpolator

g = 9.81
air_density = 1.225

# Load motor efficiency map (your confirmed structure)
# Speed axis = first column of PMSM.Speed (151 values, constant across each row)
# Torque axis = first row of PMSM.Torque (30 values, constant down each column)
# Efficiency map = PMSM.Efficiency (151 x 30)

file_path=('motor_data2.xlsx') 
xls = pd.ExcelFile(file_path)   # <<< THIS WAS MISSING

def sheet_to_numeric(sheet_name: str) -> np.ndarray:
    arr = pd.read_excel(xls, sheet_name=sheet_name, header=None)
    arr = arr.apply(pd.to_numeric, errors="coerce").to_numpy()
    return arr

speed_grid = sheet_to_numeric("PMSM.Speed")
torque_grid = sheet_to_numeric("PMSM.Torque")
eff_map_raw = sheet_to_numeric("PMSM.Efficiency")

# Extract axes exactly (based on your uniqueness test)
motor_speed_axis = speed_grid[:, 0].astype(float)   # 151 values
motor_torque_axis = torque_grid[0, :].astype(float) # 30 values
eff_map = eff_map_raw.astype(float)                 # 151 x 30

# Basic validation (helps catch hidden headers/blank borders)
if eff_map.shape[0] != len(motor_speed_axis) or eff_map.shape[1] != len(motor_torque_axis):
    raise ValueError(
        f"Shape mismatch: eff_map {eff_map.shape}, "
        f"speed_axis {len(motor_speed_axis)}, torque_axis {len(motor_torque_axis)}. "
        "Check for extra header rows/cols or blank borders in Excel.")

# Sort axes (RegularGridInterpolator requires increasing axes)
s_order = np.argsort(motor_speed_axis)
t_order = np.argsort(motor_torque_axis)

motor_speed_axis = motor_speed_axis[s_order]
motor_torque_axis = motor_torque_axis[t_order]
eff_map = eff_map[np.ix_(s_order, t_order)]

# Build interpolator (speed, torque) -> efficiency %
eff_interp = RegularGridInterpolator(
    (motor_speed_axis, motor_torque_axis),
    eff_map,
    bounds_error=False,
    fill_value=np.nan)

def get_efficiency_percent(speed_radps: float, torque_nm: float) -> float:
    """Returns motor efficiency in percent (0-100)."""
    # Most maps are provided for +torque only
    tq = abs(float(torque_nm))
    sp = float(speed_radps)

    # Clip to map domain to avoid unstable extrapolation
    sp = float(np.clip(sp, motor_speed_axis[0], motor_speed_axis[-1]))
    tq = float(np.clip(tq, motor_torque_axis[0], motor_torque_axis[-1]))

    val = eff_interp([[sp, tq]])[0]
    if np.isnan(val):
        return 85.0  # safe fallback (%)
    return float(np.clip(val, 1.0, 99.0))

# Physics-based energy calculation
def calculate_energy(
    mass,
    rolling_resistance_coefficient,
    drag_coefficient,
    frontal_area,
    slope_percent,
    wheel_diameter,
    reduction_ratio,
    drivetrain_efficiency,
    velocity,
    battery_efficiency,
    distance):
    # Safety
    velocity = max(float(velocity), 0.05)  # avoid division by zero
    drivetrain_efficiency = float(np.clip(drivetrain_efficiency, 0.01, 0.99))
    battery_efficiency = float(np.clip(battery_efficiency, 0.01, 0.99))
    reduction_ratio = max(float(reduction_ratio), 0.01)
    wheel_diameter = max(float(wheel_diameter), 0.01)

    # Time
    time_s = float(distance) / velocity

    # Forces
    rolling_force = float(rolling_resistance_coefficient) * float(mass) * g
    drag_force = 0.5 * float(drag_coefficient) * float(frontal_area) * air_density * velocity**2
    grade_angle = math.atan(float(slope_percent) / 100.0)
    gradient_force = float(mass) * g * math.sin(grade_angle)

    # Wheel force demand (+ means motoring demand)
    tractive_force = rolling_force + drag_force + gradient_force

    # Wheel/axle
    wheel_radius = wheel_diameter / 2.0
    axle_torque = tractive_force * wheel_radius

    wheel_speed_radps = velocity / wheel_radius
    motor_speed_radps = wheel_speed_radps * reduction_ratio
    motor_speed_rpm = motor_speed_radps * (60.0 / (2.0 * np.pi))

    # Torque relationship + drivetrain efficiency handling
    # (This is a common and reasonable convention)
    if tractive_force >= 0:  # motoring
        motor_torque = axle_torque / (reduction_ratio * drivetrain_efficiency)
    else:  # regen
        motor_torque = (axle_torque * drivetrain_efficiency) / reduction_ratio

    # Mechanical power at motor shaft (can be negative during regen)
    mechanical_power_w = motor_torque * motor_speed_radps

    # Motor efficiency from map
    eta_motor = get_efficiency_percent(motor_speed_rpm, motor_torque) / 100.0
    eta_motor = float(np.clip(eta_motor, 0.01, 0.99))

    # Electrical power
    # Convention used here:
    # - motoring: electrical = mechanical / eta
    # - regen:    electrical = mechanical * eta (mechanical is negative -> electrical negative)
    if tractive_force >= 0:
        electrical_power_w = mechanical_power_w / eta_motor
        net_power_kw = electrical_power_w / 1000.0
    else:
        electrical_power_w = mechanical_power_w * eta_motor
        # battery efficiency applied on regen charging side (keeps sign negative)
        net_power_kw = (electrical_power_w * battery_efficiency) / 1000.0

    net_energy_kwh = net_power_kw * (time_s / 3600.0)

    return {
        "mass": mass,
        "rolling_resistance_coefficient": rolling_resistance_coefficient,
        "drag_coefficient": drag_coefficient,
        "frontal_area": frontal_area,
        "slope_percent": slope_percent,
        "wheel_diameter": wheel_diameter,
        "reduction_ratio": reduction_ratio,
        "drivetrain_efficiency": drivetrain_efficiency,
        "velocity": velocity,
        "distance": distance,
        "battery_efficiency": battery_efficiency,

        # "time_s": time_s,
        # "rolling_force": rolling_force,
        # "drag_force": drag_force,
        # "gradient_force": gradient_force,
        # "tractive_force": tractive_force,

        # "axle_torque": axle_torque,
        # "motor_torque": motor_torque,
        # "motor_speed_rpm": motor_speed_rpm,

        # "mechanical_power_w": mechanical_power_w,
        "motor_efficiency": eta_motor,
        # "electrical_power_w": electrical_power_w,
        # "net_power_kw": net_power_kw,
        "net_energy_kwh": net_energy_kwh,}

# -------------------------
# Generate dataset
# -------------------------
num_samples = 50000
rng = np.random.default_rng(42)

results = []
for _ in range(num_samples):
    row = calculate_energy(
        mass=int(rng.integers(40000, 111000)),
        rolling_resistance_coefficient=float(rng.uniform(0.03, 0.05)),
        drag_coefficient=float(rng.uniform(0.7, 0.9)),
        frontal_area=float(rng.uniform(15, 25)),
        slope_percent=float(rng.uniform(-14.3, 14.3)),  # <-- recommended realistic bound
        wheel_diameter=float(rng.uniform(2.0, 3.0)),
        reduction_ratio=float(rng.uniform(30, 40)),
        drivetrain_efficiency=float(rng.uniform(0.8, 0.9)),
        velocity=float(rng.uniform(2.77, 5.55)),
        battery_efficiency=float(rng.uniform(0.8, 0.9)),
        distance=float(rng.uniform(10, 5000)),)
    results.append(row)

df = pd.DataFrame(results)
output_path = ("C:/Users/a1871541/Box/Paper1/Repeat_paperwork/vary distance/try_NN/data_new_zeroeff2.csv")  
df.to_csv(output_path, index=False)
print(f"Results saved to {output_path}")"